# Text Preprocessing for NLP

Essential text preprocessing techniques for natural language processing.

## Learning Objectives

- Clean and normalize text data
- Perform tokenization
- Apply stemming and lemmatization
- Remove stopwords
- Handle special characters and encoding

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

plt.style.use('seaborn-v0_8-whitegrid')
print("NLTK ready!")

## 1. Sample Text Data

In [ ]:
# Sample texts for preprocessing
sample_texts = [
    "Hello World! This is a SAMPLE text with numbers 123 and symbols @#$.",
    "Natural Language Processing (NLP) is a sub-field of AI. It's really exciting!!!",
    "The cats are running faster than the dogs. They've been training hard.",
    "Visit https://example.com for more info! Email: user@email.com",
    "I can't believe it's not butter! :) #NLP #MachineLearning"
]

for i, text in enumerate(sample_texts):
    print(f"Text {i+1}: {text}\n")

## 2. Basic Text Cleaning

In [ ]:
def basic_clean(text):
    """Basic text cleaning."""
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

print("=== Basic Cleaning ===")
for text in sample_texts[:3]:
    print(f"Original: {text}")
    print(f"Cleaned:  {basic_clean(text)}")
    print()

## 3. Tokenization

In [ ]:
# Word tokenization
text = "Natural Language Processing (NLP) is a sub-field of AI. It's really exciting!"

print("=== Tokenization Methods ===")
print(f"Original: {text}\n")

# Simple split
simple_tokens = text.split()
print(f"Simple split ({len(simple_tokens)} tokens):")
print(simple_tokens)

# NLTK word_tokenize
nltk_tokens = word_tokenize(text)
print(f"\nNLTK tokenize ({len(nltk_tokens)} tokens):")
print(nltk_tokens)

# Sentence tokenization
sentences = sent_tokenize(text)
print(f"\nSentences ({len(sentences)}):")
for sent in sentences:
    print(f"  - {sent}")

In [ ]:
# Custom tokenizer with regex
def custom_tokenize(text):
    """Custom tokenizer handling contractions and special cases."""
    # Expand common contractions
    contractions = {
        "n't": " not", "'re": " are", "'s": " is",
        "'d": " would", "'ll": " will", "'ve": " have",
        "'m": " am"
    }
    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
    
    # Tokenize on word boundaries
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

text = "I can't believe it's not butter! They've been running."
print(f"Original: {text}")
print(f"Custom tokens: {custom_tokenize(text)}")

## 4. Stopword Removal

In [ ]:
# English stopwords
stop_words = set(stopwords.words('english'))

print(f"Number of stopwords: {len(stop_words)}")
print(f"\nSample stopwords: {list(stop_words)[:20]}")

def remove_stopwords(tokens):
    """Remove stopwords from token list."""
    return [token for token in tokens if token.lower() not in stop_words]

text = "The quick brown fox jumps over the lazy dog."
tokens = word_tokenize(text.lower())
filtered = remove_stopwords(tokens)

print(f"\nOriginal: {text}")
print(f"Tokens: {tokens}")
print(f"Without stopwords: {filtered}")

In [ ]:
# Custom stopwords
custom_stops = stop_words.copy()
custom_stops.update(['said', 'would', 'could', 'also'])  # Add more
custom_stops.discard('not')  # Keep important negation

text = "He said he would not do it, but he also could not stop."
tokens = word_tokenize(text.lower())

default_filtered = [t for t in tokens if t not in stop_words]
custom_filtered = [t for t in tokens if t not in custom_stops]

print(f"Original: {text}")
print(f"Default filter: {default_filtered}")
print(f"Custom filter: {custom_filtered}")

## 5. Stemming

In [ ]:
# Porter Stemmer
stemmer = PorterStemmer()

words = ['running', 'runs', 'ran', 'runner', 'easily', 'fairly',
         'cats', 'troubling', 'troubled', 'troubles']

print("=== Stemming ===")
print(f"{'Word':<15} {'Stem':<15}")
print("-" * 30)
for word in words:
    print(f"{word:<15} {stemmer.stem(word):<15}")

## 6. Lemmatization

In [ ]:
# WordNet Lemmatizer
lemmatizer = WordNetLemmatizer()

words = ['running', 'runs', 'ran', 'better', 'cats', 
         'geese', 'studies', 'studying', 'was', 'were']

print("=== Lemmatization ===")
print(f"{'Word':<15} {'Stem':<15} {'Lemma (v)':<15} {'Lemma (n)':<15}")
print("-" * 60)
for word in words:
    stem = stemmer.stem(word)
    lemma_v = lemmatizer.lemmatize(word, pos='v')  # verb
    lemma_n = lemmatizer.lemmatize(word, pos='n')  # noun
    print(f"{word:<15} {stem:<15} {lemma_v:<15} {lemma_n:<15}")

In [ ]:
# Smart lemmatization with POS tagging
from nltk.corpus import wordnet

def get_wordnet_pos(tag):
    """Map POS tag to WordNet POS."""
    tag_dict = {
        'J': wordnet.ADJ,
        'N': wordnet.NOUN,
        'V': wordnet.VERB,
        'R': wordnet.ADV
    }
    return tag_dict.get(tag[0], wordnet.NOUN)

def smart_lemmatize(text):
    """Lemmatize with POS-aware approach."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    
    lemmas = []
    for word, tag in tagged:
        wn_pos = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word.lower(), pos=wn_pos)
        lemmas.append(lemma)
    
    return lemmas

text = "The cats are running faster than the dogs were running."
print(f"Original: {text}")
print(f"Smart lemmatized: {smart_lemmatize(text)}")

## 7. Complete Preprocessing Pipeline

In [ ]:
class TextPreprocessor:
    """Complete text preprocessing pipeline."""
    
    def __init__(self, 
                 lowercase=True,
                 remove_urls=True,
                 remove_numbers=True,
                 remove_punctuation=True,
                 remove_stopwords=True,
                 stemming=False,
                 lemmatization=True):
        
        self.lowercase = lowercase
        self.remove_urls = remove_urls
        self.remove_numbers = remove_numbers
        self.remove_punctuation = remove_punctuation
        self.remove_stopwords_flag = remove_stopwords
        self.stemming = stemming
        self.lemmatization = lemmatization
        
        self.stop_words = set(stopwords.words('english'))
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
    
    def preprocess(self, text):
        """Apply full preprocessing pipeline."""
        if self.lowercase:
            text = text.lower()
        
        if self.remove_urls:
            text = re.sub(r'https?://\S+|www\.\S+', '', text)
        
        if self.remove_numbers:
            text = re.sub(r'\d+', '', text)
        
        if self.remove_punctuation:
            text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        if self.remove_stopwords_flag:
            tokens = [t for t in tokens if t not in self.stop_words]
        
        if self.stemming:
            tokens = [self.stemmer.stem(t) for t in tokens]
        elif self.lemmatization:
            tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        
        return tokens
    
    def preprocess_batch(self, texts):
        """Preprocess multiple texts."""
        return [self.preprocess(text) for text in texts]

# Demo
preprocessor = TextPreprocessor()

print("=== Preprocessing Pipeline ===")
for text in sample_texts:
    tokens = preprocessor.preprocess(text)
    print(f"Original: {text[:50]}...")
    print(f"Tokens: {tokens}\n")

## 8. N-grams

In [ ]:
from nltk import ngrams

def get_ngrams(tokens, n):
    """Generate n-grams from tokens."""
    return list(ngrams(tokens, n))

text = "The quick brown fox jumps over the lazy dog"
tokens = text.lower().split()

print(f"Tokens: {tokens}\n")

for n in [1, 2, 3]:
    n_grams = get_ngrams(tokens, n)
    print(f"{n}-grams ({len(n_grams)}):")
    print(n_grams[:5])
    print()

## 9. Key Takeaways

1. **Lowercase** normalizes text but may lose meaning (e.g., "US" vs "us")
2. **Tokenization** splits text into words or sentences
3. **Stopwords** are common words with little semantic value
4. **Stemming** is fast but aggressive ("running" → "run")
5. **Lemmatization** produces valid words ("better" → "good")
6. **N-grams** capture word sequences and context

### Preprocessing Checklist
- [ ] Normalize case
- [ ] Remove URLs, emails, HTML
- [ ] Handle special characters
- [ ] Tokenize appropriately
- [ ] Remove stopwords (task-dependent)
- [ ] Apply stemming or lemmatization
- [ ] Consider n-grams for context